# 멀티모달 RAG 실습

PDF에서 **텍스트 · 표 · 이미지**를 추출하고, 각 요소를 요약해 다중 벡터 검색기로 색인한 뒤 질의하는 전체 파이프라인을 실행합니다.

**흐름:** `extract` → `summarize` → `retriever(index)` → `query` → `RAG 답변`

> 실행 전: 루트의 `.env.example` 을 `.env` 로 복사하고 `OPENAI_API_KEY` 를 채우세요.

In [10]:
# 0. 환경 설정
import sys
from pathlib import Path

# 프로젝트 루트를 import 경로에 추가 (src 패키지 사용)
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
print("ROOT =", ROOT)

ROOT = c:\workspaces\multimodal-rag2


## 1. PDF 요소 추출

In [11]:
from src.extract import extract_pdf_elements

PDF_PATH = ROOT / "data" / "invest.pdf"
FIGURES_DIR = ROOT / "data" / "figures"

elements = extract_pdf_elements(PDF_PATH, FIGURES_DIR)
print(elements.summary())

ModuleNotFoundError: No module named 'unstructured_inference'

## 2. 요소 요약 (텍스트 · 표 · 이미지)

In [ ]:
from src.summarize import summarize_texts, summarize_images

text_summaries, table_summaries = summarize_texts(elements.texts, elements.tables)
image_summaries = summarize_images(elements.images)

print("텍스트 요약:", len(text_summaries))
print("표 요약:", len(table_summaries))
print("이미지 요약:", len(image_summaries))

## 3. 다중 벡터 검색기 색인

In [ ]:
from src.retriever import build_retriever, index_elements

retriever = build_retriever()
index_elements(
    retriever,
    text_summaries=text_summaries,
    texts=elements.texts,
    table_summaries=table_summaries,
    tables=elements.tables,
    image_summaries=image_summaries,
    images=elements.images,
)
print("색인 완료")

## 4. 검색 및 RAG 답변

In [ ]:
from src.retriever import query

question = "이 보고서의 핵심 투자 포인트는 무엇인가요?"
docs = query(retriever, question, k=4)
for d in docs:
    print("-", str(d)[:200], "...")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "아래 컨텍스트만 근거로 질문에 한국어로 답하세요.\n\n컨텍스트:\n{context}\n\n질문: {question}"
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

context = "\n\n".join(str(d) for d in docs)
chain = prompt | llm | StrOutputParser()
print(chain.invoke({"context": context, "question": question}))